# Introduction to modelling: unbiased transmission

A simple Wright–Fisher model of two cultural traits.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(context="paper", style="ticks", 
        font_scale=1, palette="colorblind")

# random number generator for reproducibility
rng = np.random.default_rng(101101)

## 1. One generation

Assume a population of $N$ individuals. Each individual has cultural trait **A** or **B**. Let's assume that they are equally likely to have either trait. We can simulate the initial distribution of cultural traits by randomly sampling $N$ individuals from the population with replacement.

In [ ]:
N = 100
p_init = 0.5
items = ["A", "B"]

population = rng.choice(items, size=N, p=[p_init, 1 - p_init])
population[:20]

Since we're sampling, the proportion of trait A will not be exactly 50%, but it will be close. Let's look st the current mean:

In [ ]:
p_A = np.mean(population == "A")
p_A

Now we can add the next generation. We assume that with each generation, the population size remains constant, but new individuals are born to replace those that die. The individuals in the next generation copy the traits from the previous generation. Copying is random and unbiased: every individual is equally likely to be copied. 

We can simulate copying by simply sampling with replacement from the previous generation. This is similar to the agents essentially copying proportionally to the frequency of each trait in the previous generation. We can repeat this process for many generations and see how the proportion of trait A changes over time. 

If we want to sample one generation, it will look like this:

In [ ]:
population = rng.choice(population, size=N, replace=True)
np.mean(population == "A")

## 2. Multiple generations

To simulate multiple generations, we repeat the same process many-many times. The population size stays fixed, generations do not overlap, and there is no selection or innovation. This is also known as the Wright–Fisher model. 

In [ ]:
N = 100 # number of individuals in the population
t = 50 # number of generations to simulate

population = rng.choice(items, size=N, p=[p_init, 1 - p_init])
percent_A = [np.mean(population == "A")]

for _ in range(t):
    # TODO: sample the next generation with replacement
    # TODO: record the new proportion of trait A
    pass

# TODO: plot percentage of trait A against generation number
plt.show()

The proportion changes every generation even though neither trait is preferred. This random change is called **random drift**. Random drift is a stochastic process where a frequency of trait changes due to agents copying traits proportionally to their frequency in the population. Note that this type of copying is unbiased -- there is no systematic preference for traits A or B. 

## 3. Putting the model in a function

Let's put the same steps in a function. Each element of the population is one agent's trait. We record the starting proportion, then copy and record each new generation.

In [ ]:
def simulate_run(N, p_init, t, rng):
    # TODO: simulate sampling with replacement for t generations,
    # with N agents, starting with initial proportion p_init of trait A. 
    # rng is the random number generator to use for sampling.
    pass

We can plot the frequency of trait A over time:

In [ ]:
percent_A = simulate_run(N=100, p_init=0.5, t=100, rng=rng)

plt.figure(figsize=(6, 4))
plt.plot(percent_A, color="black")
plt.xlabel("Generation")
plt.ylabel("Proportion of trait A")
plt.ylim(0, 1)
sns.despine()
plt.show()

### A binomial version

Technically, we only need the number of individuals with trait A. **What kind of distribution** could we use to simulate the same process?

In [ ]:
def simulate_run_binomial(N, p_init, t, rng):
    percent_A = [p_init]
    p_A = p_init

    for _ in range(t):
        # TODO: replace the sampling with a distribution-based
        # sampling.
        pass

    return np.array(percent_A)

Does it work the same? Which option is more convenient?

In [ ]:
percent_A = simulate_run_binomial(N=10, p_init=0.5, t=100, rng=rng)

plt.figure(figsize=(6, 4))
plt.plot(percent_A, color="black")
plt.xlabel("Generation")
plt.ylabel("Proportion of trait A")
plt.ylim(0, 1)
sns.despine()
plt.show()

## 4. Repeating the model

One run is one possible outcome. Repeating the model shows the range of possible outcomes. Let's add another parameter **num_runs** (number of runs), to simulate multiple runs of the model. You can use either model type. 

In [ ]:
def run_simulation(N, p_init, t, num_runs, seed=None):
    rng = np.random.default_rng(seed)
    results = []

    for _ in range(num_runs):
        # TODO: run the model once and add the trajectory to results
        pass

    return np.array(results)

We can plot the results of multiple runs to see how the proportion of trait A changes over time in different runs. This will give us a better understanding of the variability in outcomes due to random drift.

In [ ]:
p_init_=0.8

results = run_simulation(N=10, p_init=p_init_, t=100, num_runs=100, seed=2)

plt.figure(figsize=(6, 4))
plt.axhline(p_init_, color="red", 
            linestyle="--", linewidth=2, 
            label="Starting proportion")
plt.plot(results.T, color="grey", alpha=0.25)
plt.plot(results.mean(axis=0), color="black", linewidth=2, label="Mean")
plt.xlabel("Generation")
plt.ylabel("Proportion of trait A")
plt.ylim(0, 1)
plt.legend(frameon=False)
sns.despine()
plt.show()

Some runs reach 0 or 1. Once this happens, the trait is lost or fixed and cannot return in this model.

## 5. Population size and starting proportion

Compare the effect of $N$ and $p_{init}$.

In [ ]:
p_values = [0.1, 0.5, 0.8]
N_values = [100, 1_000, 10_000]

fig, axes = plt.subplots(
    len(p_values), len(N_values), figsize=(12, 9), sharex=True, sharey=True
)

for row, p_init in enumerate(p_values):
    for col, N in enumerate(N_values):
        # TODO: simulate and plot several runs for this (p_init, N) combination
        ax = axes[row, col]
        ax.set_title(f"$p_{{init}}$ = {p_init}, N = {N:,}")
        ax.set_ylim(0, 1)

        if row == len(p_values) - 1:
            ax.set_xlabel("Generation")
        if col == 0:
            ax.set_ylabel("Proportion of trait A")

sns.despine()
plt.tight_layout()
plt.show()

What do you observe? How does the initial frequency of trait A and the population size affect the model?

## 6. Adding mutation

Let $\mu$ be the probability that a copied trait switches: A becomes B or B becomes A. Each generation, agents copy first. Then we randomly choose which agents switch their trait. That way we can implement mutation. 

In [ ]:
def simulate_with_innovation(N, p_init, t, mu, rng):
    population = rng.choice(["A", "B"], size=N, p=[p_init, 1 - p_init])
    percent_A = [np.mean(population == "A")]

    for _ in range(t):
        population = rng.choice(population, size=N, replace=True)
        # TODO: choose which individuals innovate with probability mu
        # TODO: switch A to B and B to A for those individuals
        percent_A.append(np.mean(population == "A"))

    return np.array(percent_A)

Now plot the result of a few runs, what has changed?

In [ ]:
# TODO: simulate and plot several runs with various rates of mutation.

How can you explain the current behaviour of the model? What does mutation add?

### A binomial version

We can also combine copying and flipping into one sampling step in the approach where we are sampling from a distribution. You need to find a way to recompute $p_A$ so that it will reflect the mutation rate $\mu$. 

<!-- $$p_A^* = p_A(1-\mu) + (1-p_A)\mu.$$ -->



In [ ]:
def simulate_with_innovation_binomial(N, p_init, t, mu, rng):
    percent_A = [p_init]
    p_A = p_init

    for _ in range(t):
        # TODO: implement the mutation step using the distribution plus
        # p_A recomputation with respect to mu. 
        pass

    return np.array(percent_A)

Does it work the same?

In [ ]:
# TODO: simulate and plot several runs with innovation